In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pymongo import MongoClient

# 1. Temizlenmiş veriyi okuyalım
df = pd.read_csv('/content/drive/MyDrive/Buyuk_Veri_Donem_Projesi/twitter_big_data_pipeline/data/processed/cleaned_tweets.csv')

# 2. MongoDB'ye bağlanma (Yerel kurulum varsayılan port)
try:
    client = MongoClient('mongodb://localhost:27017/')
    print("MongoDB bağlantısı başarılı!")

    # Veritabanı ve koleksiyon oluşturma
    db = client['twitter_db']
    collection = db['tweetler']

    # Eğer koleksiyonda eski veri varsa temizleyelim (kodu tekrar çalıştırırsak veri çifte kayıt olmasın)
    collection.delete_many({})

    # 3. Pandas DataFrame'i sözlük (dictionary) listesine çevirip MongoDB'ye toplu yükleme (bulk insert)
    records = df.to_dict('records')
    collection.insert_many(records)
    print(f"{len(records)} adet doküman MongoDB'ye başarıyla yüklendi.")

except Exception as e:
    print(f"Bir hata oluştu: {e}")

MongoDB bağlantısı başarılı!
14604 adet doküman MongoDB'ye başarıyla yüklendi.


MongoDB Şema Tasarımı ve Tercihi:
Bu projede ilişkisel bir veritabanı (SQL) yerine MongoDB kullanıldığı için veriyi "Flat Document" (düz doküman) mimarisinde modelledim. Tweet metni, kullanıcı bilgisi, havayolu şirketi ve duygu analizi sonuçlarının hepsi tek bir JSON benzeri doküman (BSON) içerisinde tutulmaktadır. Veriler arasında karmaşık "join" işlemleri gerektiren bir ilişki olmadığı için bu yapı, okuma (read) performansını maksimize etmektedir.

In [ ]:
print("--- 1. find() Sorguları ---")

# Sorgu 1: Sadece Virgin America şirketine ait pozitif tweetleri bulma (Koşullu filtreleme ve Limit)
print("\nSorgu 1: Virgin America'ya ait 2 pozitif tweet:")
sorgu1 = collection.find(
    {"airline": "Virgin America", "airline_sentiment": "positive"},
    {"text": 1, "airline_sentiment": 1, "_id": 0} # Projeksiyon: Sadece metin ve duygu gelsin
).limit(2)
for doc in sorgu1:
    print(doc)

# Sorgu 2: Güven skoru (confidence) 0.9'dan büyük olan negatif tweetler (Operatör kullanımı)
print("\nSorgu 2: Güven skoru çok yüksek olan negatif 2 tweet:")
sorgu2 = collection.find(
    {"airline_sentiment": "negative", "airline_sentiment_confidence": {"$gt": 0.9}},
    {"airline": 1, "negativereason": 1, "_id": 0}
).limit(2)
for doc in sorgu2:
    print(doc)

# Sorgu 3: Belirli alanları atlayarak (skip) veri getirme
print("\nSorgu 3: İlk 5 kaydı atlayıp (skip) sonraki 1 kaydı getirme:")
sorgu3 = collection.find({}, {"text": 1, "_id": 0}).skip(5).limit(1)
for doc in sorgu3:
    print(doc)

--- 1. find() Sorguları ---

Sorgu 1: Virgin America'ya ait 2 pozitif tweet:
{'airline_sentiment': 'positive', 'text': "@VirginAmerica plus you've added commercials to the experience... tacky."}
{'airline_sentiment': 'positive', 'text': '@VirginAmerica yes, nearly every time I fly VX this â\x80\x9cear wormâ\x80\x9d wonâ\x80\x99t go away :)'}

Sorgu 2: Güven skoru çok yüksek olan negatif 2 tweet:
{'negativereason': 'Bad Flight', 'airline': 'Virgin America'}
{'negativereason': "Can't Tell", 'airline': 'Virgin America'}

Sorgu 3: İlk 5 kaydı atlayıp (skip) sonraki 1 kaydı getirme:
{'text': "@VirginAmerica seriously would pay $30 a flight for seats that didn't have this playing.\nit's really the only bad thing about flying VA"}


In [ ]:
print("--- 2. Aggregation Pipeline Sorguları ---")

# Sorgu 4: Havayolu şirketlerine göre toplam tweet sayısını bulma ($group)
print("\nSorgu 4: Şirket bazlı tweet sayıları:")
pipeline1 = [
    {"$group": {"_id": "$airline", "toplam_tweet": {"$sum": 1}}},
    {"$sort": {"toplam_tweet": -1}} # Büyükten küçüğe sırala
]
for doc in collection.aggregate(pipeline1):
    print(doc)

# Sorgu 5: Sadece negatif tweetleri filtreleyip, şikayet sebeplerine göre gruplama ($match, $group)
print("\nSorgu 5: En çok rastlanan negatif tweet sebepleri:")
pipeline2 = [
    {"$match": {"airline_sentiment": "negative"}},
    {"$group": {"_id": "$negativereason", "sebep_sayisi": {"$sum": 1}}},
    {"$sort": {"sebep_sayisi": -1}},
    {"$limit": 3} # Sadece en yüksek 3 sebebi getir
]
for doc in collection.aggregate(pipeline2):
    print(doc)

# Sorgu 6: Duygu durumuna göre ortalama güven skorlarını (confidence) bulma
print("\nSorgu 6: Duygu durumuna göre ortalama güven skoru:")
pipeline3 = [
    {"$group": {
        "_id": "$airline_sentiment",
        "ortalama_guven": {"$avg": "$airline_sentiment_confidence"}
    }}
]
for doc in collection.aggregate(pipeline3):
    print(doc)

--- 2. Aggregation Pipeline Sorguları ---

Sorgu 4: Şirket bazlı tweet sayıları:
{'_id': 'United', 'toplam_tweet': 3822}
{'_id': 'US Airways', 'toplam_tweet': 2913}
{'_id': 'American', 'toplam_tweet': 2723}
{'_id': 'Southwest', 'toplam_tweet': 2420}
{'_id': 'Delta', 'toplam_tweet': 2222}
{'_id': 'Virgin America', 'toplam_tweet': 504}

Sorgu 5: En çok rastlanan negatif tweet sebepleri:
{'_id': 'Customer Service Issue', 'sebep_sayisi': 2904}
{'_id': 'Late Flight', 'sebep_sayisi': 1660}
{'_id': "Can't Tell", 'sebep_sayisi': 1190}

Sorgu 6: Duygu durumuna göre ortalama güven skoru:
{'_id': 'negative', 'ortalama_guven': 0.9332270881100557}
{'_id': 'neutral', 'ortalama_guven': 0.8228460368812682}
{'_id': 'positive', 'ortalama_guven': 0.8715497026338148}


In [ ]:
print("--- 3. İndeks, Update ve Delete İşlemleri ---")

# Sorgu 7: İndeks Oluşturma ve explain() testi
print("\nSorgu 7: 'airline' alanı için indeks oluşturuluyor...")
collection.create_index([("airline", 1)])
# İndeksin çalıştığını kanıtlamak için explain kullanımı
aciklama = collection.find({"airline": "Delta"}).explain()
print("Kullanılan indeks:", aciklama['queryPlanner']['winningPlan']['inputStage']['indexName'])

# Sorgu 8: updateOne() ile tek kayıt güncelleme ($set operatörü)
print("\nSorgu 8: İlk bulunan kayda 'incelendi: true' alanı ekleniyor...")
collection.update_one(
    {"airline_sentiment": "neutral"},
    {"$set": {"incelendi": True}}
)
ornek_update1 = collection.find_one({"incelendi": True}, {"text": 1, "incelendi": 1, "_id": 0})
print("Güncellenen kayıt:", ornek_update1)

# Sorgu 9: updateMany() ile çoklu kayıt güncelleme ($set kullanımı)
print("\nSorgu 9: Retweet sayısı 0 olanlara yeni bir alan ekleniyor...")
collection.update_many(
    {"retweet_count": 0},
    {"$set": {"etkilesim_seviyesi": "dusuk"}}
)
print("Çoklu güncelleme tamamlandı.")

# Sorgu 10: deleteMany() ile kayıt silme
print("\nSorgu 10: Güven skoru (confidence) 0.3'ten küçük olan düşük kaliteli veriler siliniyor...")
silme_sonucu = collection.delete_many({"airline_sentiment_confidence": {"$lt": 0.3}})
print(f"Silinen kayıt sayısı: {silme_sonucu.deleted_count}")

--- 3. İndeks, Update ve Delete İşlemleri ---

Sorgu 7: 'airline' alanı için indeks oluşturuluyor...
Kullanılan indeks: airline_1

Sorgu 8: İlk bulunan kayda 'incelendi: true' alanı ekleniyor...
Güncellenen kayıt: {'text': '@VirginAmerica What @dhepburn said.', 'incelendi': True}

Sorgu 9: Retweet sayısı 0 olanlara yeni bir alan ekleniyor...
Çoklu güncelleme tamamlandı.

Sorgu 10: Güven skoru (confidence) 0.3'ten küçük olan düşük kaliteli veriler siliniyor...
Silinen kayıt sayısı: 0


In [ ]:
print("--- 4. İleri Seviye Sorgular ---")

# Sorgu 11: Regex (Düzenli İfade) kullanımı
# Metni içerisinde 'baggage' (bagaj) kelimesi geçen tweetleri bulma (büyük/küçük harf duyarsız)
print("\nSorgu 11: İçinde 'baggage' geçen tweetler (Regex):")
sorgu11 = collection.find(
    {"text": {"$regex": "baggage", "$options": "i"}},
    {"airline": 1, "text": 1, "_id": 0}
).limit(2)
for doc in sorgu11:
    print(doc)

# Sorgu 12: Array operatörü kullanımı ($in)
# Sadece 'United' veya 'US Airways' havayollarına ait, sebebi 'Late Flight' (Rötar) olanları bulma
print("\nSorgu 12: United veya US Airways'e ait rötarlı (Late Flight) uçuşlar ($in operatörü):")
sorgu12 = collection.find(
    {
        "airline": {"$in": ["United", "US Airways"]},
        "negativereason": "Late Flight"
    },
    {"airline": 1, "negativereason": 1, "_id": 0}
).limit(2)
for doc in sorgu12:
    print(doc)

--- 4. İleri Seviye Sorgular ---

Sorgu 11: İçinde 'baggage' geçen tweetler (Regex):
{'airline': 'Virgin America', 'text': '@VirginAmerica hi I just booked a flight but need to add baggage, how can I do this?'}
{'airline': 'Virgin America', 'text': '@VirginAmerica Is it normal to receive no reply from Central Baggage #baggageissues #smh'}

Sorgu 12: United veya US Airways'e ait rötarlı (Late Flight) uçuşlar ($in operatörü):
{'negativereason': 'Late Flight', 'airline': 'US Airways'}
{'negativereason': 'Late Flight', 'airline': 'US Airways'}
